# 04 – Hyperparameter Tuning and Evaluation (Stage 6)
* Tuning: RandomizedSearchCV, 3-fold stratified CV, scoring = macro-F1, on a 40,000-row
  sample of the TRAINING set (compute budget). Test set untouched.
* Before/after: default CV score vs tuned CV score.
* Final fit on the full training set (200,000 rows), ONE evaluation on the held-out test set.

In [ ]:
import sys
from pathlib import Path
# make the shared module in ../src importable (no personal paths)
sys.path.append(str(Path.cwd().parent / 'src'))

In [ ]:
import json, time
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve, precision_recall_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from common import *

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
train = pd.read_csv(ROOT / "data" / "processed" / "train.csv", low_memory=False)
test = pd.read_csv(ROOT / "data" / "processed" / "test.csv", low_memory=False)
feats, num, cat = split_columns(train, True)
sub, _ = train_test_split(train, train_size=40000, stratify=train[TARGET], random_state=RANDOM_STATE)
cv = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)


def pre():
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), num),
                              ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                                ("o", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), cat)])


def pipe(model):
    return Pipeline([("pre", pre()), ("model", model)])

## 1. Search spaces (documented)

In [ ]:
SPACES = {
    "Decision Tree": (DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
                      {"model__max_depth": randint(5, 16), "model__min_samples_leaf": randint(10, 120),
                       "model__min_samples_split": randint(20, 200)}, 12),
    "Random Forest": (RandomForestClassifier(class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
                      {"model__n_estimators": [100, 150, 200], "model__max_depth": [10, 12, 14, 16],
                       "model__min_samples_leaf": [10, 20, 40], "model__max_features": ["sqrt", 0.3, 0.5]}, 10),
    "Gradient Boosting (HGB)": (HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
                                {"model__learning_rate": [0.03, 0.05, 0.1], "model__max_iter": [100, 200, 300],
                                 "model__max_depth": [3, 5, 7, None], "model__min_samples_leaf": [20, 50, 100],
                                 "model__l2_regularization": [0.0, 1.0, 10.0]}, 10),
}
DEFAULTS = {"Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
            "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
            "Gradient Boosting (HGB)": HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE)}

## 2. Tune every model and record before / after CV scores

In [ ]:
tuned, rows = {}, []
for name, (est, space, n_iter) in SPACES.items():
    t0 = time.time()
    default_cv = cross_val_score(pipe(DEFAULTS[name]), sub[feats], sub[TARGET], cv=cv, scoring="f1_macro").mean()
    rs = RandomizedSearchCV(pipe(est), space, n_iter=n_iter, scoring="f1_macro", cv=cv, random_state=RANDOM_STATE, n_jobs=1, refit=True)
    rs.fit(sub[feats], sub[TARGET])
    tuned[name] = rs
    rows.append({"model": name, "default_cv_macro_f1": round(default_cv, 4), "tuned_cv_macro_f1": round(rs.best_score_, 4),
                 "gain": round(rs.best_score_ - default_cv, 4),
                 "best_params": json.dumps({k.replace("model__", ""): (v.item() if hasattr(v, "item") else v) for k, v in rs.best_params_.items()}),
                 "n_candidates": n_iter, "minutes": round((time.time() - t0) / 60, 1)})
    print(rows[-1], flush=True)
tune_df = pd.DataFrame(rows); tune_df.to_csv(RES_DIR / "tuning_results.csv", index=False)

## 3. Final models on the FULL training set -> evaluate once on the held-out test set

In [ ]:
y_tr, y_te = train[TARGET], test[TARGET]
risk_te = y_te.isin(AT_RISK).astype(int)
final, metr, store = {}, [], {}
for name, rs in tuned.items():
    params = {k.replace("model__", ""): v for k, v in rs.best_params_.items()}
    model = pipe(rs.best_estimator_.named_steps["model"].__class__(**{**rs.best_estimator_.named_steps["model"].get_params()}))
    model.fit(train[feats], y_tr)
    final[name] = model
    pred = model.predict(test[feats]); proba = model.predict_proba(test[feats])
    classes = list(model.named_steps["model"].classes_)
    p_risk = proba[:, [classes.index(c) for c in AT_RISK]].sum(axis=1)
    store[name] = {"pred": pred, "proba": proba, "classes": classes, "p_risk": p_risk}
    metr.append({"model": name, "accuracy": accuracy_score(y_te, pred), "macro_f1": f1_score(y_te, pred, average="macro"),
                 "weighted_f1": f1_score(y_te, pred, average="weighted"),
                 "macro_auc_ovr": roc_auc_score(y_te, proba, multi_class="ovr", labels=classes),
                 "at_risk_precision": precision_score(risk_te, p_risk >= 0.5), "at_risk_recall": recall_score(risk_te, p_risk >= 0.5),
                 "at_risk_f1": f1_score(risk_te, p_risk >= 0.5), "at_risk_auc": roc_auc_score(risk_te, p_risk)})
    print(metr[-1], flush=True)
m = pd.DataFrame(metr); m.to_csv(RES_DIR / "test_metrics.csv", index=False)

# baseline on test for reference
base_acc = (y_te == y_tr.mode()[0]).mean()
json.dump({"baseline_accuracy": float(base_acc), "baseline_macro_f1": float(f1_score(y_te, [y_tr.mode()[0]] * len(y_te), average="macro"))},
          open(RES_DIR / "baseline_test.json", "w"))

## 4. Choose the final model
Rule fixed BEFORE looking at the test set: highest tuned CV macro-F1 (selection on validation folds, not on test).

In [ ]:
best_name = tune_df.sort_values("tuned_cv_macro_f1", ascending=False).iloc[0]["model"]
print("FINAL MODEL:", best_name)
S, best = store[best_name], final[best_name]
classes = S["classes"]

# per-class report
rep = pd.DataFrame(classification_report(y_te, S["pred"], labels=CLASS_ORDER, output_dict=True)).T.round(4)
rep.to_csv(RES_DIR / "classification_report_final.csv"); print(rep)

# confusion matrices (4-class counts + row-normalised)
cm = confusion_matrix(y_te, S["pred"], labels=CLASS_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, data, fmt, ttl in [(axes[0], cm, "d", "Counts"), (axes[1], cm / cm.sum(axis=1, keepdims=True), ".2f", "Row-normalised (recall)")]:
    im = ax.imshow(data, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_ORDER, rotation=25); ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_ORDER)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, format(data[i, j], fmt), ha="center", va="center", color="white" if data[i, j] > data.max() / 2 else "black", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"{best_name}: confusion matrix – {ttl}")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig09_confusion_matrix.png"); plt.close()
pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER).to_csv(RES_DIR / "confusion_matrix_final.csv")

# at-risk ROC and PR curves for ALL tuned models
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for name in store:
    fpr, tpr, _ = roc_curve(risk_te, store[name]["p_risk"]); axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(risk_te, store[name]['p_risk']):.3f})")
    pr, rc, _ = precision_recall_curve(risk_te, store[name]["p_risk"]); axes[1].plot(rc, pr, label=name)
axes[0].plot([0, 1], [0, 1], "k--", lw=.8); axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC – at-risk (Fail + Withdrawn) detection"); axes[0].legend(frameon=False, fontsize=8)
axes[1].axhline(risk_te.mean(), color="k", ls="--", lw=.8); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–recall – at-risk detection"); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig10_roc_pr_atrisk.png"); plt.close()

# model comparison bar chart
fig, ax = plt.subplots(figsize=(9, 4.4)); w = .2; x = np.arange(len(m))
for i, (col, lab) in enumerate([("accuracy", "Accuracy"), ("macro_f1", "Macro-F1"), ("at_risk_recall", "At-risk recall"), ("at_risk_auc", "At-risk AUC")]):
    ax.bar(x + (i - 1.5) * w, m[col], w, label=lab)
ax.set_xticks(x); ax.set_xticklabels(m["model"]); ax.set_ylim(0, 1.12); ax.legend(frameon=False, ncol=4, loc="upper center")
ax.set_title("Tuned models on the held-out test set"); plt.tight_layout(); plt.savefig(FIG_DIR / "fig11_model_comparison.png"); plt.close()

# before / after tuning chart
fig, ax = plt.subplots(figsize=(7.5, 4)); x = np.arange(len(tune_df))
ax.bar(x - .18, tune_df.default_cv_macro_f1, .36, label="Default", color="#9AA5B1"); ax.bar(x + .18, tune_df.tuned_cv_macro_f1, .36, label="Tuned", color="#2E6FA3")
for i, r in tune_df.iterrows(): ax.text(i + .18, r.tuned_cv_macro_f1 + .004, f"{r.tuned_cv_macro_f1:.3f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(tune_df.model); ax.set_ylim(.4, .7); ax.set_ylabel("CV macro-F1"); ax.legend(frameon=False)
ax.set_title("Effect of hyperparameter tuning (3-fold CV)"); plt.tight_layout(); plt.savefig(FIG_DIR / "fig12_tuning_before_after.png"); plt.close()

## 5. Business metrics: threshold, capture rate and lift for the at-risk score

In [ ]:
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    flag = S["p_risk"] >= t
    rows.append({"threshold": t, "flagged_pct": flag.mean() * 100, "precision": precision_score(risk_te, flag),
                 "recall": recall_score(risk_te, flag), "f1": f1_score(risk_te, flag)})
thr = pd.DataFrame(rows).round(3); thr.to_csv(RES_DIR / "threshold_table.csv", index=False); print(thr)
order = np.argsort(-S["p_risk"]); rk = risk_te.values[order]
cap = pd.DataFrame({"top_pct": [10, 20, 30, 40]})
cap["capture_pct"] = [rk[: int(len(rk) * p / 100)].sum() / rk.sum() * 100 for p in cap.top_pct]
cap["lift"] = cap.capture_pct / cap.top_pct; cap.round(2).to_csv(RES_DIR / "capture_lift.csv", index=False); print(cap)

## 6. Feature importance (permutation importance on 10,000 test rows, macro-F1 drop)

In [ ]:
ts = test.sample(10000, random_state=RANDOM_STATE)
pi = permutation_importance(best, ts[feats], ts[TARGET], scoring="f1_macro", n_repeats=3, random_state=RANDOM_STATE, n_jobs=1)
imp = pd.DataFrame({"feature": feats, "importance": pi.importances_mean, "std": pi.importances_std}).sort_values("importance", ascending=False)
imp.to_csv(RES_DIR / "feature_importance.csv", index=False); print(imp.head(12))
top = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5.2)); ax.barh(top.feature, top.importance, xerr=top["std"], color="#2E6FA3")
ax.set_xlabel("Drop in macro-F1 when feature is shuffled"); ax.set_title(f"Top 15 features – {best_name}")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig13_feature_importance.png"); plt.close()

## 7. Save the final model + metadata for the Streamlit app

In [ ]:
medians = train[num].median().to_dict()
modes = {c: train[c].mode()[0] for c in cat}
meta = {"model_name": best_name, "features": feats, "numeric": num, "categorical": cat, "classes": classes,
        "medians": {k: float(v) for k, v in medians.items()}, "modes": modes,
        "choices": {c: sorted(train[c].dropna().unique().tolist()) for c in cat},
        "ranges": {c: [float(train[c].min()), float(train[c].max())] for c in num},
        "best_params": tune_df.set_index("model").loc[best_name, "best_params"],
        "test_macro_f1": float(m.set_index("model").loc[best_name, "macro_f1"]),
        "test_accuracy": float(m.set_index("model").loc[best_name, "accuracy"]),
        "at_risk_auc": float(m.set_index("model").loc[best_name, "at_risk_auc"])}
joblib.dump(best, MODEL_DIR / "student_performance_model.joblib", compress=3)
json.dump(meta, open(MODEL_DIR / "model_metadata.json", "w"), indent=2)
size = (MODEL_DIR / "student_performance_model.joblib").stat().st_size / 1e6
print(f"Saved model ({size:.1f} MB)")